In [ ]:
import h5py
with h5py.File("./data/baryons_logPkR.h5", 'r') as f:
    print(f.keys())
    print(f['antilles'].keys())

In [ ]:
import numpy as np
import scipy

nsrcs = 8
nlens = 10
ntheta = 20
ndv = (nsrcs*(nsrcs+1)+nsrcs*nlens+nlens)*ntheta

baseline  = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/chains/roman_baseline_evaluate/roman_baseline.modelvector')[:,1]
                        
pollution = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/chains/roman_pollution_evaluate/roman_pollution.modelvector')[:,1]
print(f'len of baseline {len(baseline)}; len of polution {len(pollution)}; expected ndv {ndv}')

cov_raw = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/data/cov_roman')
cov = np.zeros((ndv,ndv))
for i in range(len(cov_raw)):
    ii,jj = int(cov_raw[i,0]),int(cov_raw[i,1])
    component = cov_raw[i,8]+cov_raw[i,9]
    cov[ii,jj] = component
    cov[jj,ii] = component

#mask = np.ones(ndv)
mask_full = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/data/roman_full_shear_DESY3_lens.mask')[:,1]
tol = 0.5/20

def probechi2(start,end,probe):
    delta = (pollution - baseline)[start:end]
    mask = mask_full[start:end].astype(bool)
    delta_masked = delta[mask]
    cov_delta = cov[start:end,:][:,start:end]
    cov_delta_masked = cov_delta[mask,:][:,mask]
    invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
    chi2 = delta_masked@invcov_delta_masked@delta_masked
    print(f'chi2 of probe {probe} is {chi2}')
    print(f'{np.sum(mask)}/{len(mask)} points remain for probe {probe}')

ncombo0=0
cnt = 0
for ni in range(nsrcs):
    for nj in range(ni, nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        idx_mask=0
        while(chi2>tol):
            mask_full[cnt*ntheta+idx_mask] = 0
            mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
            delta_masked = delta[mask]
            cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
            cov_delta_masked = cov_delta[mask,:][:,mask]
            invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
            chi2 = delta_masked@invcov_delta_masked@delta_masked
            idx_mask+=1
        print(f'xi+ ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo1=cnt
probechi2(ncombo0*ntheta, ncombo1*ntheta, 'xi+')
for ni in range(nsrcs):
    for nj in range(ni, nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        idx_mask=0
        while(chi2>tol):
            mask_full[cnt*ntheta+idx_mask] = 0
            mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
            delta_masked = delta[mask]
            cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
            cov_delta_masked = cov_delta[mask,:][:,mask]
            invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
            chi2 = delta_masked@invcov_delta_masked@delta_masked
            idx_mask+=1
        print(f'xi- ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo2=cnt
probechi2(ncombo1*ntheta, ncombo2*ntheta,'xi-')
for ni in range(nlens):
    for nj in range(nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        print(f'gammat ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo3=cnt
probechi2(ncombo2*ntheta, ncombo3*ntheta,'gammat')
for ni in range(nlens):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        print(f'wtheta ni={ni}, nj={ni}, chi2={chi2}')
        cnt+=1
ncombo4=cnt
probechi2(ncombo3*ntheta, ncombo4*ntheta,'wtheta')

len of baseline 3240; len of polution 3240; expected ndv 3240
xi+ ni=0, nj=0, chi2=0.0073330272664990604
xi+ ni=0, nj=1, chi2=0.007354067570641627
xi+ ni=0, nj=2, chi2=0.00596983774996235
xi+ ni=0, nj=3, chi2=0.005088042083770991
xi+ ni=0, nj=4, chi2=0.004388893679723754
xi+ ni=0, nj=5, chi2=0.003779332902714638
xi+ ni=0, nj=6, chi2=0.0031821177025881855
xi+ ni=0, nj=7, chi2=0.002485423377638916
xi+ ni=1, nj=1, chi2=0.01193380445046649
xi+ ni=1, nj=2, chi2=0.018107874028998397
xi+ ni=1, nj=3, chi2=0.017512713009938057
xi+ ni=1, nj=4, chi2=0.01685457733768126
xi+ ni=1, nj=5, chi2=0.015859332504381908
xi+ ni=1, nj=6, chi2=0.014411685009224011
xi+ ni=1, nj=7, chi2=0.01213584816132136
xi+ ni=2, nj=2, chi2=0.013913995959645072
xi+ ni=2, nj=3, chi2=0.023767356346226863
xi+ ni=2, nj=4, chi2=0.02281963318408138
xi+ ni=2, nj=5, chi2=0.022218466888244966
xi+ ni=2, nj=6, chi2=0.020748287269120695
xi+ ni=2, nj=7, chi2=0.02316271663462586
xi+ ni=3, nj=3, chi2=0.020463242143440204
xi+ ni=3, nj=4, ch

In [2]:
nlen = len(mask_full)
out = np.zeros((nlen,2))
out[:,0] = np.arange(nlen)
out[:,1] = mask_full
print(f'check the number {np.sum(mask_full)}/{len(mask_full)} for this mask')
np.savetxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/data/roman_Y3.mask', out)

check the number 1614.0/3240 for this mask


### this is for len=source

In [1]:
import numpy as np
import scipy

nsrcs = 8
nlens = 8
ntheta = 20
ggl_exclude= [(6, 0), (7, 0), (7, 1)]
ndv = (nsrcs*(nsrcs+1)+nsrcs*nlens+nlens-len(ggl_exclude))*ntheta

baseline  = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/chains/roman_les_evaluate/roman_les.modelvector')[:,1]
                        
pollution = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/chains/roman_les_pollution_evaluate/roman_les_pollution.modelvector')[:,1]
print(f'len of baseline {len(baseline)}; len of polution {len(pollution)}; expected ndv {ndv}')

cov_raw = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/data/cov_roman_les')
cov = np.zeros((ndv,ndv))
for i in range(len(cov_raw)):
    ii,jj = int(cov_raw[i,0]),int(cov_raw[i,1])
    component = cov_raw[i,8]+cov_raw[i,9]
    cov[ii,jj] = component
    cov[jj,ii] = component

#mask = np.ones(ndv)
mask_full = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/roman_real/data/roman_les_full_shear_DESY3_lens.mask')[:,1]
tol = 0.5/20

def probechi2(start,end,probe):
    delta = (pollution - baseline)[start:end]
    mask = mask_full[start:end].astype(bool)
    delta_masked = delta[mask]
    cov_delta = cov[start:end,:][:,start:end]
    cov_delta_masked = cov_delta[mask,:][:,mask]
    invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
    chi2 = delta_masked@invcov_delta_masked@delta_masked
    print(f'chi2 of probe {probe} is {chi2}')
    print(f'{np.sum(mask)}/{len(mask)} points remain for probe {probe}')

ncombo0=0
cnt = 0
for ni in range(nsrcs):
    for nj in range(ni, nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        idx_mask=0
        while(chi2>tol):
            mask_full[cnt*ntheta+idx_mask] = 0
            mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
            delta_masked = delta[mask]
            cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
            cov_delta_masked = cov_delta[mask,:][:,mask]
            invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
            chi2 = delta_masked@invcov_delta_masked@delta_masked
            idx_mask+=1
        print(f'xi+ ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo1=cnt
probechi2(ncombo0*ntheta, ncombo1*ntheta, 'xi+')
for ni in range(nsrcs):
    for nj in range(ni, nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        idx_mask=0
        while(chi2>tol):
            mask_full[cnt*ntheta+idx_mask] = 0
            mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
            delta_masked = delta[mask]
            cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
            cov_delta_masked = cov_delta[mask,:][:,mask]
            invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
            chi2 = delta_masked@invcov_delta_masked@delta_masked
            idx_mask+=1
        print(f'xi- ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo2=cnt
probechi2(ncombo1*ntheta, ncombo2*ntheta,'xi-')
for ni in range(nlens):
    for nj in range(nsrcs):
        if (ni, nj) in ggl_exclude:
            continue
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        print(f'gammat ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo3=cnt
probechi2(ncombo2*ntheta, ncombo3*ntheta,'gammat')
for ni in range(nlens):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        print(f'wtheta ni={ni}, nj={ni}, chi2={chi2}')
        cnt+=1
ncombo4=cnt
probechi2(ncombo3*ntheta, ncombo4*ntheta,'wtheta')

len of baseline 2820; len of polution 2820; expected ndv 2820
xi+ ni=0, nj=0, chi2=0.007333205157432783
xi+ ni=0, nj=1, chi2=0.007354149603668271
xi+ ni=0, nj=2, chi2=0.005969882712647521
xi+ ni=0, nj=3, chi2=0.0050880811406296255
xi+ ni=0, nj=4, chi2=0.004388915473840122
xi+ ni=0, nj=5, chi2=0.0037793509231703556
xi+ ni=0, nj=6, chi2=0.00318212526855633
xi+ ni=0, nj=7, chi2=0.002485424397605823
xi+ ni=1, nj=1, chi2=0.01193417323329
xi+ ni=1, nj=2, chi2=0.018108535799891686
xi+ ni=1, nj=3, chi2=0.01751342822230465
xi+ ni=1, nj=4, chi2=0.01685531016864341
xi+ ni=1, nj=5, chi2=0.01586001651207644
xi+ ni=1, nj=6, chi2=0.014412418376042038
xi+ ni=1, nj=7, chi2=0.012136483167720591
xi+ ni=2, nj=2, chi2=0.013914654702199021
xi+ ni=2, nj=3, chi2=0.023768484622344956
xi+ ni=2, nj=4, chi2=0.02282065031204251
xi+ ni=2, nj=5, chi2=0.022219435219581894
xi+ ni=2, nj=6, chi2=0.020749233362198326
xi+ ni=2, nj=7, chi2=0.02316375932161467
xi+ ni=3, nj=3, chi2=0.02046421460739021
xi+ ni=3, nj=4, chi2=0.

In [2]:
nlen = len(mask_full)
out = np.zeros((nlen,2))
out[:,0] = np.arange(nlen)
out[:,1] = mask_full
print(f'check the number {np.sum(mask_full)}/{len(mask_full)} for this mask')
np.savetxt('./data/roman_les_Y3.mask', out)

check the number 1437.0/2820 for this mask
